# 34 - C_BLOCK runtime validation (real ABI path)

Este notebook valida o fluxo completo do `c_block` sem fallback:

1. Compila um C block com 3 saidas (seno aproximada, quadrada e triangular).
2. Valida as saidas via `CBlockLibrary.step()`.
3. Valida as mesmas saidas no backend (`Simulator.run_transient`) com canais `CB1`, `CB1.out0..out2`.
4. Confirma o contrato **run or error** (circuito sem `lib_path` deve falhar).


In [1]:
from pathlib import Path
import sys
import tempfile
import numpy as np
import matplotlib.pyplot as plt

# Resolve local build/python similarly to other project notebooks.
_root = Path.cwd()
_candidates = []
for _ in range(6):
    for _rel in (("build-test", "python"), ("build", "python")):
        candidate = _root / _rel[0] / _rel[1]
        if candidate.is_dir():
            _candidates.append(candidate)
    _root = _root.parent

_seen = set()
for candidate in _candidates:
    cstr = str(candidate)
    if cstr in _seen:
        continue
    _seen.add(cstr)
    if cstr not in sys.path:
        sys.path.insert(0, cstr)

import pulsim as ps
from pulsim import CBlockLibrary, compile_cblock, detect_compiler

cc = detect_compiler()
if not cc:
    raise RuntimeError("No C compiler found. Set PULSIM_CC to clang/gcc/msvc and rerun.")

print(f"Pulsim version: {ps.__version__}")
print(f"Compiler      : {cc}")


Pulsim version: 0.7.5
Compiler      : /usr/bin/cc


In [2]:
C_SOURCE = r'''
#include "pulsim/v1/cblock_abi.h"

PULSIM_CBLOCK_EXPORT int pulsim_cblock_abi_version = PULSIM_CBLOCK_ABI_VERSION;

PULSIM_CBLOCK_EXPORT int pulsim_cblock_step(
    PulsimCBlockCtx* ctx,
    double t,
    double dt,
    const double* in,
    double* out) {
    (void)ctx;
    (void)dt;
    (void)in;

    const double freq = 1200.0;
    const double phase = t * freq;
    const long long cycles = (long long)phase;
    const double frac = phase - (double)cycles;  // [0, 1)

    // Square: +/-1
    const double square = (frac < 0.5) ? 1.0 : -1.0;

    // Triangle: [-1, 1]
    const double triangle = 4.0 * ((frac < 0.5) ? frac : (1.0 - frac)) - 1.0;

    // Sine approximation without libm dependency.
    const double x = 6.28318530717958647692 * frac - 3.14159265358979323846; // [-pi, pi]
    const double ax = (x < 0.0) ? -x : x;
    double sine = 1.27323954473516268615 * x - 0.40528473456935108577 * x * ax; // 4/pi and 4/pi^2
    const double asine = (sine < 0.0) ? -sine : sine;
    sine = 0.225 * (sine * asine - sine) + sine;

    out[0] = sine;
    out[1] = square;
    out[2] = triangle;
    return 0;
}
'''

build_dir = Path(tempfile.mkdtemp(prefix="pulsim_cblock_nb_"))
lib_path = Path(
    compile_cblock(C_SOURCE, name="wavegen_multiout", output_dir=build_dir, compiler=cc)
)

print(f"Build dir: {build_dir}")
print(f"Library  : {lib_path}")
assert lib_path.exists()


Build dir: /var/folders/0n/gs7hh4fj4bn8r3r4qsykw2840000gn/T/pulsim_cblock_nb_k19hg77z
Library  : /private/var/folders/0n/gs7hh4fj4bn8r3r4qsykw2840000gn/T/pulsim_cblock_nb_k19hg77z/libwavegen_multiout.dylib


In [3]:
def estimate_frequency_from_rising_edges(signal: np.ndarray, time: np.ndarray) -> float:
    edges = np.where((signal[:-1] < 0.0) & (signal[1:] >= 0.0))[0]
    if len(edges) < 2:
        return 0.0
    duration = float(time[edges[-1] + 1] - time[edges[0]])
    if duration <= 0.0:
        return 0.0
    return (len(edges) - 1) / duration

# 1) Direct ABI stepping
n_direct = 4000
t_direct = np.linspace(0.0, 4e-3, n_direct)
y_direct = np.zeros((n_direct, 3), dtype=float)

with CBlockLibrary(lib_path, n_inputs=1, n_outputs=3, name="wavegen") as blk:
    prev_t = 0.0
    for i, t in enumerate(t_direct):
        dt = 0.0 if i == 0 else float(t - prev_t)
        y_direct[i, :] = blk.step(float(t), dt, [0.0])
        prev_t = float(t)

sine_d = y_direct[:, 0]
square_d = y_direct[:, 1]
tri_d = y_direct[:, 2]

assert sine_d.max() > 0.8 and sine_d.min() < -0.8
assert square_d.max() > 0.5 and square_d.min() < -0.5
assert tri_d.max() > 0.9 and tri_d.min() < -0.9

f_est_direct = estimate_frequency_from_rising_edges(square_d, t_direct)
assert abs(f_est_direct - 1200.0) / 1200.0 < 0.05

print(f"Direct check OK. Estimated frequency: {f_est_direct:.2f} Hz")


Direct check OK. Estimated frequency: 1199.70 Hz


In [4]:
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(t_direct * 1e3, sine_d, lw=1.3)
ax[0].set_ylabel("sine")
ax[0].grid(True, alpha=0.3)

ax[1].plot(t_direct * 1e3, square_d, lw=1.3)
ax[1].set_ylabel("square")
ax[1].grid(True, alpha=0.3)

ax[2].plot(t_direct * 1e3, tri_d, lw=1.3)
ax[2].set_ylabel("triangle")
ax[2].set_xlabel("time [ms]")
ax[2].grid(True, alpha=0.3)

fig.suptitle("C_BLOCK outputs via CBlockLibrary.step")
plt.tight_layout()
display(fig)
plt.close(fig)


<Figure size 1100x700 with 3 Axes>

In [5]:
# 2) Backend transient with c_block virtual component
ckt = ps.Circuit()
n_in = ckt.add_node("in")
gnd = ckt.ground()

ckt.add_voltage_source("Vin", n_in, gnd, 0.0)
ckt.add_resistor("Rin", n_in, gnd, 1e3)
ckt.add_virtual_component(
    "c_block",
    "CB1",
    [n_in],
    {"n_inputs": 1.0, "n_outputs": 3.0},
    {"lib_path": str(lib_path)},
)

opts = ps.SimulationOptions()
opts.tstart = 0.0
opts.tstop = 4e-3
opts.dt = 2e-6
opts.step_mode = ps.StepMode.Fixed
opts.dt_min = opts.dt
opts.dt_max = opts.dt
opts.control_mode = ps.ControlUpdateMode.Continuous
opts.control_sample_time = 0.0
opts.newton_options.num_nodes = ckt.num_nodes()
opts.newton_options.num_branches = ckt.num_branches()

result = ps.Simulator(ckt, opts).run_transient(ckt.initial_state())
assert result.success, result.message

time = np.asarray(result.time, dtype=float)
channels = result.virtual_channels

required = ["CB1", "CB1.out0", "CB1.out1", "CB1.out2"]
for ch in required:
    assert ch in channels, f"Missing channel: {ch}"

cb = np.asarray(channels["CB1"], dtype=float)
out0 = np.asarray(channels["CB1.out0"], dtype=float)
out1 = np.asarray(channels["CB1.out1"], dtype=float)
out2 = np.asarray(channels["CB1.out2"], dtype=float)

assert len(cb) == len(time)
assert len(out0) == len(time)
assert len(out1) == len(time)
assert len(out2) == len(time)

# Canonical mirror: base channel == out0
assert np.allclose(cb, out0)

assert out0.max() > 0.8 and out0.min() < -0.8
assert out1.max() > 0.5 and out1.min() < -0.5
assert out2.max() > 0.9 and out2.min() < -0.9

f_est_backend = estimate_frequency_from_rising_edges(out1, time)
assert abs(f_est_backend - 1200.0) / 1200.0 < 0.10

print(f"Transient check OK. Estimated frequency: {f_est_backend:.2f} Hz")
print(f"Samples: {len(time)}")


Transient check OK. Estimated frequency: 1199.04 Hz
Samples: 2001


In [6]:
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(time * 1e3, out0, lw=1.2)
ax[0].set_ylabel("CB1.out0")
ax[0].grid(True, alpha=0.3)

ax[1].plot(time * 1e3, out1, lw=1.2)
ax[1].set_ylabel("CB1.out1")
ax[1].grid(True, alpha=0.3)

ax[2].plot(time * 1e3, out2, lw=1.2)
ax[2].set_ylabel("CB1.out2")
ax[2].set_xlabel("time [ms]")
ax[2].grid(True, alpha=0.3)

fig.suptitle("C_BLOCK outputs via Simulator.run_transient")
plt.tight_layout()
display(fig)
plt.close(fig)


<Figure size 1100x700 with 3 Axes>

In [7]:
# 3) Fail-fast contract: missing lib_path must raise runtime error
ckt_fail = ps.Circuit()
nf = ckt_fail.add_node("in")
gf = ckt_fail.ground()
ckt_fail.add_voltage_source("Vin", nf, gf, 0.0)
ckt_fail.add_resistor("Rin", nf, gf, 1e3)
ckt_fail.add_virtual_component(
    "c_block",
    "CB_FAIL",
    [nf],
    {"n_inputs": 1.0, "n_outputs": 1.0},
    {},
)

opts_fail = ps.SimulationOptions()
opts_fail.tstart = 0.0
opts_fail.tstop = 2e-5
opts_fail.dt = 1e-6
opts_fail.step_mode = ps.StepMode.Fixed
opts_fail.dt_min = opts_fail.dt
opts_fail.dt_max = opts_fail.dt
opts_fail.control_mode = ps.ControlUpdateMode.Continuous
opts_fail.control_sample_time = 0.0
opts_fail.newton_options.num_nodes = ckt_fail.num_nodes()
opts_fail.newton_options.num_branches = ckt_fail.num_branches()

try:
    ps.Simulator(ckt_fail, opts_fail).run_transient(ckt_fail.initial_state())
    raise AssertionError("Expected runtime error for missing c_block lib_path")
except RuntimeError as exc:
    msg = str(exc)
    assert "lib_path" in msg or "C_BLOCK" in msg
    print("Fail-fast check OK:")
    print(msg)


Fail-fast check OK:
C_BLOCK 'CB_FAIL' missing required 'lib_path'


## Resultado esperado

Se todas as celulas executarem sem erro, voce validou:

- geracao multi-sinal no C block (seno/quadrada/triangular);
- exportacao de canais no backend (`CB1`, `CB1.out0`, `CB1.out1`, `CB1.out2`);
- consistencia de comprimento com `result.time`;
- contrato **run or error** quando falta implementacao real (`lib_path`).
